# Lab 11: 1D CNN for Time-Series Forecasting

**Course:** Machine Learning Lab  
**Topic:** One-dimensional convolutional neural network for AEP forecasting

## Lab Objective
The objective of this lab is to train a 1D CNN on time-series windows. The model uses convolution layers to learn local temporal patterns and predict the target value.


## 1. Set Working Directory
The working directory is changed so the notebook can access the dataset, helper functions, checkpoints, and saved history files.


In [1]:
import os
os.chdir(r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code')

## 2. Import Libraries and Utilities
This cell imports forecasting metrics, sequence preparation functions, Keras layers, callbacks, plotting tools, and scaling utilities.


In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

## 3. Define Initial Parameters
The notebook initializes the model state, start epoch, number of time steps, and number of features for the 1D CNN input.


In [3]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

## 4. Define the 1D CNN Architecture
The CNN uses 1D convolution layers to extract temporal features from a sequence window, then flattens those features and predicts one output value.


In [4]:
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

## 5. Display Model Summary and Diagram
The summary and model plot confirm the CNN layer sequence, tensor shapes, and number of trainable parameters.


In [5]:
model1 = CNN()
model1.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 24, 21)]          0         
                                                                 
 conv1d (Conv1D)             (None, 23, 16)            688       
                                                                 
 conv1d_1 (Conv1D)           (None, 22, 16)            528       
                                                                 
 flatten (Flatten)           (None, 352)               0         
                                                                 
 dense (Dense)               (None, 1)                 353       
                                                                 
Total params: 1569 (6.13 KB)
Trainable params: 1569 (6.13 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [6]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


## 6. Set Checkpoint and History Paths
These paths control where the best model checkpoint, history plot, and history JSON file are saved.


In [7]:
checkpoints = r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

## 7. Configure Callbacks
Callbacks save the best model according to validation loss and record the training history.


In [8]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

## 8. Compile or Load the CNN
A new CNN is compiled when no checkpoint is supplied. If a checkpoint is provided, training can continue from the saved model.


In [9]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =CNN()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## 9. Load Dataset Files
The train, validation, and test CSV files are loaded and converted into arrays for sequence generation. The scaler is loaded for inverse transformation during evaluation.


In [10]:
import os
path_dataset =r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

## 10. Prepare Time-Series Input Windows
The time step and feature count are confirmed, then the data is converted into supervised-learning sequences for training, validation, and testing.


In [11]:
time_steps=24
num_features=21

In [12]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.013962745666503906 sec


## 11. Train the 1D CNN
The CNN is trained using the prepared sequence windows and monitored with validation data after each epoch.


In [13]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/10
27/27 [==============================] - ETA: 0s - loss: 0.3185 - mae: 0.3185 - mape: 173.8397
Epoch 1: val_loss improved from inf to 0.11304, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.11.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 5s 73ms/step - loss: 0.3185 - mae: 0.3185 - mape: 173.8397 - val_loss: 0.1130 - val_mae: 0.1130 - val_mape: 34.3522
Epoch 2/10
26/27 [===========================>..] - ETA: 0s - loss: 0.1044 - mae: 0.1044 - mape: 59.4824
Epoch 2: val_loss improved from 0.11304 to 0.07314, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0002-loss0.07.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 34ms/step - loss: 0.1044 - mae: 0.1044 - mape: 59.4027 - val_loss: 0.0731 - val_mae: 0.0731 - val_mape: 20.9882
Epoch 3/10
20/27 [=====================>........] - ETA: 0s - loss: 0.0814 - mae: 0.0814 - mape: 45.3540
Epoch 3: val_loss improved from 0.07314 to 0.06191, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0003-loss0.06.h5


...

27/27 [==============================] - 1s 39ms/step - loss: 0.0694 - mae: 0.0694 - mape: 37.2700 - val_loss: 0.0533 - val_mae: 0.0533 - val_mape: 17.7505
Epoch 5/10
26/27 [===========================>..] - ETA: 0s - loss: 0.0639 - mae: 0.0639 - mape: 32.2843
Epoch 5: val_loss improved from 0.05332 to 0.05129, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0005-loss0.05.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 42ms/step - loss: 0.0641 - mae: 0.0641 - mape: 32.2520 - val_loss: 0.0513 - val_mae: 0.0513 - val_mape: 17.3098
Epoch 6/10
24/27 [=========================>....] - ETA: 0s - loss: 0.0612 - mae: 0.0612 - mape: 32.0694
Epoch 6: val_loss did not improve from 0.05129
27/27 [==============================] - 1s 34ms/step - loss: 0.0618 - mae: 0.0618 - mape: 32.2495 - val_loss: 0.0668 - val_mae: 0.0668 - val_mape: 21.8131
Epoch 7/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0603 - mae: 0.0603 - mape: 28.4676
Epoch 7: val_loss did not improve from 0.05129
27/27 [==============================] - 1s 29ms/step - loss: 0.0600 - mae: 0.0600 - mape: 29.7529 - val_loss: 0.0604 - val_mae: 0.0604 - val_mape: 19.5677
Epoch 8/10
19/27 [====================>.........] - ETA: 0s - loss: 0.0583 - mae: 0.0583 - mape: 27.4799
Epoch 8: val_loss did not improve from 0.05129
27/27 [==============================] - 1s 31ms/step - loss: 0.0552 - mae: 

c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 35ms/step - loss: 0.0638 - mae: 0.0638 - mape: 29.7876 - val_loss: 0.0457 - val_mae: 0.0457 - val_mape: 16.2986
Epoch 10/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0570 - mae: 0.0570 - mape: 26.1807
Epoch 10: val_loss did not improve from 0.04575
27/27 [==============================] - 1s 30ms/step - loss: 0.0565 - mae: 0.0565 - mape: 26.0667 - val_loss: 0.0755 - val_mae: 0.0755 - val_mape: 24.2624


## 12. Evaluate Test Performance
The saved CNN checkpoint predicts test values. Predictions are inverse-transformed and evaluated using MAE, RMSE, MAPE, and R2.


In [14]:

model = load_model(r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0008-loss0.03.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 286ms/step
Mean Absolute Error (MAE): 1368.97
Median Absolute Error (MedAE): 1339.72
Mean Squared Error (MSE): 1978978.9
Root Mean Squared Error (RMSE): 1406.76
Mean Absolute Percentage Error (MAPE): 8.76 %
Median Absolute Percentage Error (MDAPE): 8.68 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## 13. Fine-Tuning Setup
This section selects a previous checkpoint and starting epoch so the model can continue training.


In [15]:
checkpoints = r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0008-loss0.03.h5'
model=r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0008-loss0.03.h5'
start_epoch= 58

## 14. Rebuild Callbacks and Load Checkpoint
The model and callbacks are prepared again for the fine-tuning stage.


In [16]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0008-loss0.03.h5...
[INFO] old learning rate: 9.999999747378752e-05
[INFO] new learning rate: 9.999999747378752e-05


## 15. Continue Training
Additional epochs are used to improve or refine the CNN model performance.


In [17]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0483 - mae: 0.0483 - mape: 24.4973
Epoch 1: val_loss improved from inf to 0.03556, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0008-loss0.03.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 3s 52ms/step - loss: 0.0457 - mae: 0.0457 - mape: 23.0198 - val_loss: 0.0356 - val_mae: 0.0356 - val_mape: 10.6047
Epoch 2/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0453 - mae: 0.0453 - mape: 22.7785
Epoch 2: val_loss did not improve from 0.03556
27/27 [==============================] - 1s 28ms/step - loss: 0.0450 - mae: 0.0450 - mape: 23.0308 - val_loss: 0.0357 - val_mae: 0.0357 - val_mape: 10.7896
Epoch 3/10
22/27 [=======================>......] - ETA: 0s - loss: 0.0441 - mae: 0.0441 - mape: 22.4544
Epoch 3: val_loss did not improve from 0.03556
27/27 [==============================] - 1s 28ms/step - loss: 0.0445 - mae: 0.0445 - mape: 22.6435 - val_loss: 0.0361 - val_mae: 0.0361 - val_mape: 11.1681
Epoch 4/10
22/27 [=======================>......] - ETA: 0s - loss: 0.0439 - mae: 0.0439 - mape: 22.5496
Epoch 4: val_loss did not improve from 0.03556
27/27 [==============================] - 1s 28ms/step - loss: 0.0442 - mae: 

## 16. Evaluate Fine-Tuned CNN
The fine-tuned checkpoint is evaluated on the same test set so its performance can be compared with the earlier model.


In [18]:

model = load_model(r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0008-loss0.03.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 201ms/step
Mean Absolute Error (MAE): 1470.0
Median Absolute Error (MedAE): 1442.29
Mean Squared Error (MSE): 2267002.87
Root Mean Squared Error (RMSE): 1505.66
Mean Absolute Percentage Error (MAPE): 9.41 %
Median Absolute Percentage Error (MDAPE): 9.35 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Conclusion
This lab shows that 1D CNNs can learn temporal patterns from fixed-length time-series windows. Checkpointing and fine-tuning help preserve the best model and continue improving performance.
